In [2]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
import category_encoders as ce
import gc

DATA_FOLDER ='/mnt/d/NF-CICIDS2018-v3'


In [2]:
df = dd.read_csv(Path(DATA_FOLDER) / 'NF-CICIDS2018-v3.csv')

df.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1518611287705,1518611287705,172.31.0.2,53,172.31.66.58,63593,17,5,156,1,...,0,0,0,0,0,0,0,0,0,Benign
1,1518611287743,1518611290747,172.31.66.58,56163,239.255.255.250,1900,17,12,805,5,...,0,2793,750,1182,0,0,0,0,0,Benign
2,1518611288143,1518611300202,172.31.66.46,62388,239.255.255.250,1900,17,12,1288,8,...,0,3016,1722,1436,0,0,0,0,0,Benign
3,1518611288165,1518611336194,0.0.0.0,546,0.0.0.0,547,17,103,393,3,...,16014,32014,24014,8000,0,0,0,0,0,Benign
4,1518611288175,1518611288176,172.31.66.46,49187,169.254.169.254,80,6,7,373,5,...,0,0,0,0,0,0,0,0,0,Benign


In [3]:
df = df.sort_values('FLOW_START_MILLISECONDS')

# time split
split_millisecs = 20 * 1000  # milliseconds
start_time = df['FLOW_START_MILLISECONDS'].min().compute()
df['time_group'] = (df['FLOW_START_MILLISECONDS'] - start_time) // split_millisecs

# construct src and dst, drop unnecessary columns
df['src'] = df['IPV4_SRC_ADDR'].astype('str') + ':' + df['L4_SRC_PORT'].astype('str')
df['dst'] = df['IPV4_DST_ADDR'].astype('str') + ':' + df['L4_DST_PORT'].astype('str')
df = df.drop(columns=['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'DNS_QUERY_ID', 'Label' ])

cols = ['src', 'dst'] + [col for col in df.columns if col not in ['src', 'dst']]
df = df[cols]

# inf and values can occur, possibly due to 0 durations -> fill with 0s
nan_cols = ['SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES']
df[nan_cols] = df[nan_cols].replace([np.inf, -np.inf], np.nan)
df[nan_cols] = df[nan_cols].fillna(0)

# 'label' column, and 'normal' label for the benign data is expected by the model 
df = df.rename(columns={'Attack': 'label'})
label_mapping = {
    'Bot': 'Bot',
    'Brute_Force_-Web': 'Web',
    'Brute_Force_-XSS': 'Web',
    'SQL_Injection': 'Web',
    'DDOS_attack-HOIC': 'DDoS',
    'DDOS_attack-LOIC-UDP': 'DDoS',
    'DDoS_attacks-LOIC-HTTP': 'DDoS',
    'DoS_attacks-GoldenEye': 'DoS',
    'DoS_attacks-Hulk': 'DoS',
    'DoS_attacks-SlowHTTPTest': 'DoS',
    'DoS_attacks-Slowloris': 'DoS',
    'FTP-BruteForce': 'Bruteforce',
    'SSH-Bruteforce': 'Bruteforce',
    'Infilteration': 'Infiltration',
    'Benign': 'normal'
}

df['label'] = df['label'].map_partitions(lambda s: s.replace(label_mapping))

df.head(1000)

,src,dst,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,label,time_group
0,172.31.0.2:53,172.31.66.58:63593,17,5,156,1,0,0,0,0,...,0,0,0,0,0,0,0,0,normal,0
1,172.31.66.58:56163,239.255.255.250:1900,17,12,805,5,0,0,0,0,...,0,2793,750,1182,0,0,0,0,normal,0
2,172.31.66.46:62388,239.255.255.250:1900,17,12,1288,8,0,0,0,0,...,0,3016,1722,1436,0,0,0,0,normal,0
3,0.0.0.0:546,0.0.0.0:547,17,103,393,3,0,0,0,0,...,16014,32014,24014,8000,0,0,0,0,normal,0
4,172.31.66.46:49187,169.254.169.254:80,6,7,373,5,700,5,27,27,...,0,0,0,0,0,0,0,0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,172.31.67.91:55645,172.31.0.2:53,17,5,67,1,104,1,0,0,...,0,0,0,0,0,0,0,0,normal,1
996,172.31.69.6:49773,74.125.28.106:443,6,91,921,10,3875,8,223,223,...,0,175,63,64,0,350,73,118,normal,1
997,172.31.69.6:49772,74.125.28.106:443,6,91,2904,26,5691,29,218,218,...,0,58351,4697,15769,0,58311,4190,14982,normal,1
998,172.31.69.6:49775,74.125.28.106:443,6,91,961,11,3914,9,219,219,...,0,172,63,55,0,317,71,101,normal,1


In [4]:
# check for inf and nan values
numeric_df = df.select_dtypes(include=[np.number])
rows_with_invalid = numeric_df.map_partitions(lambda part: (np.isinf(part) | part.isna()).any(axis=1), meta=bool)
print(f'Number of invalid values: {int(rows_with_invalid.sum().compute())}')

Number of invalid values: 0


In [5]:
# label time groups
def label_group_pandas(group):
    unique_labels = set(group['label'].unique())
    if len(unique_labels) == 1:
        label = unique_labels.pop()
    elif 'normal' in unique_labels and len(unique_labels) == 2:
        unique_labels.remove('normal')
        label = unique_labels.pop()
    else:
        label = 'unknown'
    return pd.DataFrame({'time_group': [group.name], 'label': [label]})

label_df = df.groupby('time_group').apply(
    label_group_pandas,
    meta={'time_group': 'int64', 'label': 'object'}
).compute()

label_df = label_df.reset_index(drop=True).sort_values('time_group')

# assign each time group into train/test
def split_train_test_pandas(group):
    n = len(group)
    train_end = int(n * 0.8)
    group['type'] = ['train'] * train_end + ['test'] * (n - train_end)
    return group

labeled_with_split = label_df.groupby('label', group_keys=False).apply(split_train_test_pandas)
labeled_with_split = labeled_with_split[labeled_with_split['label'] != 'unknown']

# save output
output_path = Path(DATA_FOLDER) / "labels.csv"
labeled_with_split.to_csv(output_path, index=False)
print(f"Saved to {output_path}")


Saved to /mnt/d/NF-CICIDS2018-v3/labels.csv


/tmp/ipykernel_21704/2484909722.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labeled_with_split = label_df.groupby('label', group_keys=False).apply(split_train_test_pandas)


In [6]:
unique_labels = df['label'].unique().compute()
le = LabelEncoder()
le.fit(unique_labels)
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
df['label'] = df['label'].map(label_map, meta=('label', 'int64'))

df.head(1000)

,src,dst,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,label,time_group
0,172.31.0.2:53,172.31.66.58:63593,17,5,156,1,0,0,0,0,...,0,0,0,0,0,0,0,0,6,0
1,172.31.66.58:56163,239.255.255.250:1900,17,12,805,5,0,0,0,0,...,0,2793,750,1182,0,0,0,0,6,0
2,172.31.66.46:62388,239.255.255.250:1900,17,12,1288,8,0,0,0,0,...,0,3016,1722,1436,0,0,0,0,6,0
3,0.0.0.0:546,0.0.0.0:547,17,103,393,3,0,0,0,0,...,16014,32014,24014,8000,0,0,0,0,6,0
4,172.31.66.46:49187,169.254.169.254:80,6,7,373,5,700,5,27,27,...,0,0,0,0,0,0,0,0,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,172.31.67.91:55645,172.31.0.2:53,17,5,67,1,104,1,0,0,...,0,0,0,0,0,0,0,0,6,1
996,172.31.69.6:49773,74.125.28.106:443,6,91,921,10,3875,8,223,223,...,0,175,63,64,0,350,73,118,6,1
997,172.31.69.6:49772,74.125.28.106:443,6,91,2904,26,5691,29,218,218,...,0,58351,4697,15769,0,58311,4190,14982,6,1
998,172.31.69.6:49775,74.125.28.106:443,6,91,961,11,3914,9,219,219,...,0,172,63,55,0,317,71,101,6,1


In [7]:
le.classes_

array(['Bot', 'Bruteforce', 'DDoS', 'DoS', 'Infiltration', 'Web',
       'normal'], dtype=object)

In [8]:
idx_label_df = pd.DataFrame(le.classes_)
idx_label_df.columns = ['label']
idx_label_df.to_csv(Path(DATA_FOLDER) / 'idx_label.csv', index=True)

In [9]:
df['label'].value_counts().compute()

label
3      302966
4      188152
5        2538
0      207703
2     1324350
6    17514626
1      575194
Name: count, dtype: int64

In [10]:
df.to_csv(Path(DATA_FOLDER) / 'data-checkpoint1.csv', index=False, single_file=True)

['/mnt/d/NF-CICIDS2018-v3/data-checkpoint1.csv']

In [3]:
# in case of limited memory, reset kernel here

df = pd.read_csv(Path(DATA_FOLDER) / 'data-checkpoint1.csv')
label_df = pd.read_csv(Path(DATA_FOLDER) / 'labels.csv')

train_idx = set(label_df[label_df['type'] == 'train']['time_group'])
test_idx = set(label_df[label_df['type'] == 'test']['time_group'])

train_df = df[df['time_group'].isin(train_idx)]
test_df = df[df['time_group'].isin(test_idx)]

print(train_df.shape[0] + test_df.shape[0])

test_df.to_csv(Path(DATA_FOLDER) / 'test-data-checkpoint1.csv', index=False)

del df
del test_df
gc.collect()

20115529


0

In [4]:
categorical_cols = [
    'PROTOCOL', 'L7_PROTO', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS',
    'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE'
]
target_col = 'label'

# save memory by processing data col by col
encoders = {}
for col in categorical_cols:
    encoder = ce.TargetEncoder(cols=[col])
    encoder.fit(train_df[[col]], train_df[target_col])
    train_df[col] = encoder.transform(train_df[[col]])[col]
    encoders[col] = encoder

train_df.head(5000)

,src,dst,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,label,time_group
0,172.31.0.2:53,172.31.66.58:63593,5.973090,5.975331,156,1,0,0,5.973238,5.973238,...,0,0,0,0,0,0,0,0,6,0
1,172.31.66.58:56163,239.255.255.250:1900,5.973090,5.988855,805,5,0,0,5.973238,5.973238,...,0,2793,750,1182,0,0,0,0,6,0
2,172.31.66.46:62388,239.255.255.250:1900,5.973090,5.988855,1288,8,0,0,5.973238,5.973238,...,0,3016,1722,1436,0,0,0,0,6,0
3,0.0.0.0:546,0.0.0.0:547,5.973090,6.000000,393,3,0,0,5.973238,5.973238,...,16014,32014,24014,8000,0,0,0,0,6,0
4,172.31.66.46:49187,169.254.169.254:80,5.271383,3.996222,373,5,700,5,5.260789,5.371426,...,0,0,0,0,0,0,0,0,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,172.31.69.8:54054,172.31.0.2:53,5.973090,5.975331,78,1,206,1,5.973238,5.973238,...,0,0,0,0,0,0,0,0,6,6
4996,172.31.65.85:55843,172.31.0.2:53,5.973090,5.975331,67,1,228,1,5.973238,5.973238,...,0,0,0,0,0,0,0,0,6,6
4997,172.31.65.85:60127,172.31.0.2:53,5.973090,5.975331,70,1,130,1,5.973238,5.973238,...,0,0,0,0,0,0,0,0,6,6
4998,172.31.67.73:3389,5.101.40.105:54734,5.271383,5.994769,40,1,0,0,5.997203,5.997203,...,0,0,0,0,0,0,0,0,6,6


In [ ]:
scalers = {}

cols_to_norm = list(set(train_df.columns) - {'src', 'dst', 'label', 'time_group'})

for col in cols_to_norm:
    scaler = StandardScaler()
    scaled_col = scaler.fit_transform(train_df[[col]])
    train_df[col] = scaled_col
    scalers[col] = scaler

train_df.to_csv(Path(DATA_FOLDER) / 'train-data-checkpoint1.csv', index=False)
train_df.head(5000)

,src,dst,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,label,time_group
0,172.31.0.2:53,172.31.66.58:63593,1.225537,0.409502,-0.019979,-0.017993,-0.020866,-0.042279,0.484300,0.673639,...,-0.073419,-0.348455,-0.339618,-0.345535,-0.106231,-0.31978,-0.25377,-0.305493,6,0
1,172.31.66.58:56163,239.255.255.250:1900,1.225537,0.422577,-0.014570,-0.015945,-0.020866,-0.042279,0.484300,0.673639,...,-0.073419,-0.137525,-0.117077,-0.096536,-0.106231,-0.31978,-0.25377,-0.305493,6,0
2,172.31.66.46:62388,239.255.255.250:1900,1.225537,0.422577,-0.010544,-0.014408,-0.020866,-0.042279,0.484300,0.673639,...,-0.073419,-0.120684,0.171337,-0.043028,-0.106231,-0.31978,-0.25377,-0.305493,6,0
3,0.0.0.0:546,0.0.0.0:547,1.225537,0.433352,-0.018004,-0.016969,-0.020866,-0.042279,0.484300,0.673639,...,9.212480,2.069274,6.785859,1.339738,-0.106231,-0.31978,-0.25377,-0.305493,6,0
4,172.31.66.46:49187,169.254.169.254:80,-0.815569,-1.503875,-0.018171,-0.015945,-0.018245,-0.014882,-0.334351,-0.288236,...,-0.073419,-0.348455,-0.339618,-0.345535,-0.106231,-0.31978,-0.25377,-0.305493,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,172.31.69.8:54054,172.31.0.2:53,1.225537,0.409502,-0.020629,-0.017993,-0.020094,-0.036800,0.484300,0.673639,...,-0.073419,-0.348455,-0.339618,-0.345535,-0.106231,-0.31978,-0.25377,-0.305493,6,6
4996,172.31.65.85:55843,172.31.0.2:53,1.225537,0.409502,-0.020721,-0.017993,-0.020012,-0.036800,0.484300,0.673639,...,-0.073419,-0.348455,-0.339618,-0.345535,-0.106231,-0.31978,-0.25377,-0.305493,6,6
4997,172.31.65.85:60127,172.31.0.2:53,1.225537,0.409502,-0.020696,-0.017993,-0.020379,-0.036800,0.484300,0.673639,...,-0.073419,-0.348455,-0.339618,-0.345535,-0.106231,-0.31978,-0.25377,-0.305493,6,6
4998,172.31.67.73:3389,5.101.40.105:54734,-0.815569,0.428294,-0.020946,-0.017993,-0.020866,-0.042279,0.511838,0.711943,...,-0.073419,-0.348455,-0.339618,-0.345535,-0.106231,-0.31978,-0.25377,-0.305493,6,6


In [6]:
del train_df
gc.collect()

input_path = Path(DATA_FOLDER) / 'train-data-checkpoint1.csv'
output_path = Path(DATA_FOLDER) / 'X_train.csv'

open(output_path, 'w').close()

# Chunked processing and writing
first_chunk = True

for chunk in pd.read_csv(input_path, chunksize=100_000):
    # Compute features column
    chunk['features'] = chunk[cols_to_norm].values.tolist()

    # Select required columns
    output_chunk = chunk[['src', 'dst', 'features', 'time_group']]

    # Write to disk
    output_chunk.to_csv(output_path, mode='a', index=False, header=first_chunk)
    first_chunk = False  # only write header once

    del chunk, output_chunk
    gc.collect()

In [7]:
test_df = pd.read_csv(Path(DATA_FOLDER) / 'test-data-checkpoint1.csv')

for col in categorical_cols:
    encoder = encoders[col]
    test_df[col] = encoder.transform(test_df[[col]])[col]

for col in cols_to_norm:
    test_df[col] = scalers[col].transform(test_df[[col]])

test_df['features'] = test_df[cols_to_norm].values.tolist()

test_df.to_csv(Path(DATA_FOLDER) / 'X_test.csv', columns=['src', 'dst', 'features', 'time_group'], index=False)